# Notely AI Table Exploration

This notebook is a first-pass exploration of the simulated Notely AI analytics database.

Use it to understand:

- What tables exist in the local SQLite database.
- What each table looks like.
- How product events, subscriptions, billing, experiments, and support tickets connect.
- Whether the embedded product storylines are visible in the data.

The database lives at `data/notely_ai.sqlite`.

## 1. Connect to the Database

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

db_path = project_root / "data" / "notely_ai.sqlite"
print(db_path)
print("Database exists:", db_path.exists())

conn = sqlite3.connect(db_path)

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn)

## 2. List Tables

This is the first query to run whenever you open a new database.

In [ ]:
q("""
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""")

## 3. Row Counts by Table

This tells us the rough scale of the simulated warehouse.

In [ ]:
q("""
SELECT 'users' AS table_name, COUNT(*) AS row_count FROM users
UNION ALL SELECT 'sessions', COUNT(*) FROM sessions
UNION ALL SELECT 'events', COUNT(*) FROM events
UNION ALL SELECT 'experiments', COUNT(*) FROM experiments
UNION ALL SELECT 'subscriptions', COUNT(*) FROM subscriptions
UNION ALL SELECT 'billing_invoices', COUNT(*) FROM billing_invoices
UNION ALL SELECT 'payments', COUNT(*) FROM payments
UNION ALL SELECT 'support_tickets', COUNT(*) FROM support_tickets
UNION ALL SELECT 'product_calendar', COUNT(*) FROM product_calendar
ORDER BY row_count DESC;
""")

## 4. Inspect Table Schemas

SQLite stores schema metadata in `PRAGMA table_info(table_name)`.

In [ ]:
for table in ["users", "events", "sessions", "subscriptions", "billing_invoices", "payments", "experiments", "support_tickets", "product_calendar"]:
    print(f"\n--- {table} ---")
    display(q(f"PRAGMA table_info({table});"))

## 5. Preview Core Tables

In [ ]:
q("""
SELECT *
FROM users
LIMIT 10;
""")

In [ ]:
q("""
SELECT *
FROM events
ORDER BY event_timestamp
LIMIT 10;
""")

In [ ]:
q("""
SELECT *
FROM subscriptions
LIMIT 10;
""")

## 6. Product Calendar

This table is important for future RAG/root-cause analysis because it stores known product context: launches, incidents, pricing changes, experiments, and campaigns.

In [ ]:
q("""
SELECT *
FROM product_calendar
ORDER BY calendar_date;
""")

## 7. User Mix

Start by understanding where users come from and what personas exist.

In [ ]:
q("""
SELECT
  acquisition_channel,
  COUNT(*) AS users,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_users
FROM users
GROUP BY 1
ORDER BY users DESC;
""")

In [ ]:
q("""
SELECT
  user_segment,
  COUNT(*) AS users,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_users
FROM users
GROUP BY 1
ORDER BY users DESC;
""")

## 8. Event Volume by Event Name

This helps identify the most common product behaviors.

In [ ]:
q("""
SELECT
  event_name,
  COUNT(*) AS events,
  COUNT(DISTINCT user_id) AS users
FROM events
GROUP BY 1
ORDER BY events DESC;
""")

## 9. Monthly New Users

This is a simple growth metric that can later appear in weekly/monthly reports.

In [ ]:
monthly_new_users = q("""
SELECT
  substr(signup_date, 1, 7) AS signup_month,
  COUNT(*) AS new_users
FROM users
GROUP BY 1
ORDER BY 1;
""")

monthly_new_users

In [ ]:
monthly_new_users.plot(x="signup_month", y="new_users", kind="line", marker="o", figsize=(10, 4), title="Monthly New Users");

## 10. Activation Funnel

Activation is defined as a new user completing these actions within 7 days of signup:

- `workspace_created`
- `recording_upload_completed`
- `ai_summary_generated`

In [ ]:
q("""
WITH user_steps AS (
  SELECT
    u.user_id,
    MAX(CASE WHEN e.event_name = 'workspace_created' THEN 1 ELSE 0 END) AS has_workspace,
    MAX(CASE WHEN e.event_name = 'recording_upload_completed' THEN 1 ELSE 0 END) AS has_upload,
    MAX(CASE WHEN e.event_name = 'ai_summary_generated' THEN 1 ELSE 0 END) AS has_summary
  FROM users u
  LEFT JOIN events e
    ON u.user_id = e.user_id
   AND e.event_date BETWEEN u.signup_date AND date(u.signup_date, '+7 day')
  GROUP BY 1
)
SELECT 'signup_completed' AS step, COUNT(*) AS users FROM user_steps
UNION ALL SELECT 'workspace_created', SUM(has_workspace) FROM user_steps
UNION ALL SELECT 'recording_upload_completed', SUM(CASE WHEN has_workspace = 1 AND has_upload = 1 THEN 1 ELSE 0 END) FROM user_steps
UNION ALL SELECT 'ai_summary_generated', SUM(CASE WHEN has_workspace = 1 AND has_upload = 1 AND has_summary = 1 THEN 1 ELSE 0 END) FROM user_steps;
""")

## 11. Activation Rate by Platform and Week

This query is useful for finding the iOS upload bug storyline.

In [ ]:
weekly_activation = q("""
WITH signup_cohort AS (
  SELECT
    u.user_id,
    date(u.signup_date, 'weekday 1', '-7 days') AS signup_week,
    u.platform,
    MAX(CASE WHEN e.event_name = 'workspace_created' THEN 1 ELSE 0 END) AS has_workspace,
    MAX(CASE WHEN e.event_name = 'recording_upload_completed' THEN 1 ELSE 0 END) AS has_upload,
    MAX(CASE WHEN e.event_name = 'ai_summary_generated' THEN 1 ELSE 0 END) AS has_summary
  FROM users u
  LEFT JOIN events e
    ON u.user_id = e.user_id
   AND e.event_date BETWEEN u.signup_date AND date(u.signup_date, '+7 day')
  GROUP BY 1, 2, 3
)
SELECT
  signup_week,
  platform,
  COUNT(*) AS new_users,
  ROUND(AVG(CASE WHEN has_workspace = 1 AND has_upload = 1 AND has_summary = 1 THEN 1.0 ELSE 0.0 END), 3) AS activation_rate
FROM signup_cohort
WHERE signup_week BETWEEN '2026-04-20' AND '2026-06-01'
GROUP BY 1, 2
ORDER BY 1, 2;
""")

weekly_activation

In [ ]:
weekly_activation.pivot(index="signup_week", columns="platform", values="activation_rate").plot(figsize=(10, 4), marker="o", title="Weekly Activation Rate by Platform");

## 12. iOS Upload Bug Deep Dive

Known product context: `product_calendar` says an iOS upload retry bug started on 2026-05-10 and was resolved on 2026-05-17.

In [ ]:
q("""
SELECT
  event_date,
  platform,
  COUNT(*) AS upload_failures
FROM events
WHERE event_name = 'recording_upload_failed'
  AND event_date BETWEEN '2026-05-05' AND '2026-05-22'
GROUP BY 1, 2
ORDER BY 1, 2;
""")

In [ ]:
q("""
SELECT
  created_date,
  platform,
  COUNT(*) AS upload_failed_tickets
FROM support_tickets
WHERE category = 'upload_failed'
  AND created_date BETWEEN '2026-05-05' AND '2026-05-22'
GROUP BY 1, 2
ORDER BY 1, 2;
""")

## 13. Paid Search Quality Decline

Known product context: paid search spend increased on 2026-06-01, bringing more lower-intent users.

In [ ]:
q("""
WITH user_outcomes AS (
  SELECT
    u.user_id,
    CASE WHEN u.signup_date >= '2026-06-01' THEN 'after_2026_06_01' ELSE 'before_2026_06_01' END AS period,
    u.acquisition_channel,
    MAX(CASE WHEN e.event_name = 'ai_summary_generated' THEN 1 ELSE 0 END) AS generated_summary,
    MAX(CASE WHEN e.event_name = 'subscription_started' THEN 1 ELSE 0 END) AS became_paid
  FROM users u
  LEFT JOIN events e ON u.user_id = e.user_id
  WHERE u.signup_date BETWEEN '2026-05-01' AND '2026-06-26'
  GROUP BY 1, 2, 3
)
SELECT
  period,
  acquisition_channel,
  COUNT(*) AS signups,
  ROUND(AVG(generated_summary), 3) AS summary_generation_rate,
  ROUND(AVG(became_paid), 3) AS paid_conversion_rate
FROM user_outcomes
GROUP BY 1, 2
ORDER BY 1, signups DESC;
""")

## 14. Onboarding Flow Experiment

`onboarding_flow_v2` compares standard onboarding with a guided setup flow.

In [ ]:
q("""
WITH experiment_users AS (
  SELECT
    x.variant,
    u.user_id,
    MAX(CASE WHEN e.event_name = 'workspace_created' THEN 1 ELSE 0 END) AS has_workspace,
    MAX(CASE WHEN e.event_name = 'recording_upload_completed' THEN 1 ELSE 0 END) AS has_upload,
    MAX(CASE WHEN e.event_name = 'ai_summary_generated' THEN 1 ELSE 0 END) AS has_summary
  FROM experiments x
  JOIN users u ON x.user_id = u.user_id
  LEFT JOIN events e
    ON u.user_id = e.user_id
   AND e.event_date BETWEEN u.signup_date AND date(u.signup_date, '+7 day')
  WHERE x.experiment_name = 'onboarding_flow_v2'
  GROUP BY 1, 2
)
SELECT
  variant,
  COUNT(*) AS users,
  ROUND(AVG(has_workspace), 3) AS workspace_rate,
  ROUND(AVG(has_upload), 3) AS upload_rate,
  ROUND(AVG(CASE WHEN has_workspace = 1 AND has_upload = 1 AND has_summary = 1 THEN 1.0 ELSE 0.0 END), 3) AS activation_rate
FROM experiment_users
GROUP BY 1
ORDER BY 1;
""")

## 15. Action Items and D7 Retention

This query checks whether users who use the action items feature retain better.

In [ ]:
q("""
WITH base AS (
  SELECT
    u.user_id,
    u.signup_date,
    MAX(CASE WHEN e.event_name = 'action_items_generated' THEN 1 ELSE 0 END) AS used_action_items
  FROM users u
  LEFT JOIN events e ON u.user_id = e.user_id
  WHERE u.signup_date BETWEEN '2026-04-15' AND '2026-06-10'
  GROUP BY 1, 2
),
retention AS (
  SELECT
    b.user_id,
    b.used_action_items,
    CASE WHEN COUNT(e.event_id) > 0 THEN 1 ELSE 0 END AS retained_d7
  FROM base b
  LEFT JOIN events e
    ON b.user_id = e.user_id
   AND e.event_date BETWEEN date(b.signup_date, '+7 day') AND date(b.signup_date, '+13 day')
  GROUP BY 1, 2
)
SELECT
  used_action_items,
  COUNT(*) AS users,
  ROUND(AVG(retained_d7), 3) AS d7_retention
FROM retention
GROUP BY 1;
""")

## 16. Free Limit Change

Known product context: the Free plan monthly summary limit changed from 10 to 5 on 2026-05-25.

In [ ]:
q("""
WITH cohorts AS (
  SELECT
    u.user_id,
    CASE WHEN u.signup_date >= '2026-05-25' THEN 'after_limit_change' ELSE 'before_limit_change' END AS period,
    MAX(CASE WHEN e.event_name = 'usage_limit_reached' THEN 1 ELSE 0 END) AS hit_usage_limit,
    MAX(CASE WHEN e.event_name = 'paywall_viewed' THEN 1 ELSE 0 END) AS saw_paywall,
    MAX(CASE WHEN e.event_name = 'subscription_started' THEN 1 ELSE 0 END) AS became_paid
  FROM users u
  LEFT JOIN events e ON u.user_id = e.user_id
  WHERE u.signup_date BETWEEN '2026-05-01' AND '2026-06-20'
  GROUP BY 1, 2
)
SELECT
  period,
  COUNT(*) AS users,
  ROUND(AVG(hit_usage_limit), 3) AS usage_limit_rate,
  ROUND(AVG(saw_paywall), 3) AS paywall_view_rate,
  ROUND(AVG(became_paid), 3) AS paid_conversion_rate
FROM cohorts
GROUP BY 1;
""")

## 17. Weekly PM Report Starter Metrics

This query gives a first draft of weekly metrics that could power the automated PM report.

In [ ]:
q("""
WITH weekly_users AS (
  SELECT
    date(signup_date, 'weekday 1', '-7 days') AS week_start,
    COUNT(*) AS new_users
  FROM users
  GROUP BY 1
),
weekly_events AS (
  SELECT
    date(event_date, 'weekday 1', '-7 days') AS week_start,
    COUNT(DISTINCT CASE WHEN event_name = 'recording_upload_completed' THEN event_id END) AS completed_uploads,
    COUNT(DISTINCT CASE WHEN event_name = 'ai_summary_generated' THEN event_id END) AS summaries_generated,
    COUNT(DISTINCT CASE WHEN event_name = 'subscription_started' THEN user_id END) AS new_paid_users,
    COUNT(DISTINCT user_id) AS active_users
  FROM events
  GROUP BY 1
)
SELECT
  wu.week_start,
  wu.new_users,
  we.active_users,
  we.completed_uploads,
  ROUND(1.0 * we.completed_uploads / NULLIF(we.active_users, 0), 2) AS uploads_per_active_user,
  we.summaries_generated,
  ROUND(1.0 * we.summaries_generated / NULLIF(we.active_users, 0), 2) AS summaries_per_active_user,
  we.new_paid_users
FROM weekly_users wu
LEFT JOIN weekly_events we ON wu.week_start = we.week_start
WHERE wu.week_start >= '2026-04-01'
ORDER BY wu.week_start;
""")

## 18. Billing Persistence

Billing persistence is a subscription health metric. Here, we approximate it as the share of invoices that are successfully paid.

In [ ]:
q("""
SELECT
  substr(invoice_date, 1, 7) AS invoice_month,
  COUNT(*) AS invoices,
  SUM(CASE WHEN status = 'paid' THEN 1 ELSE 0 END) AS paid_invoices,
  ROUND(AVG(CASE WHEN status = 'paid' THEN 1.0 ELSE 0.0 END), 3) AS billing_persistence
FROM billing_invoices
GROUP BY 1
ORDER BY 1;
""")

## 19. Close Connection

In [ ]:
conn.close()